In [289]:
import numpy as np
import qutip as qtp
from qiskit.quantum_info import SparsePauliOp, Pauli, Statevector
from qiskit import QuantumCircuit
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from itertools import combinations
import networkx as nx
from qiskit_ibm_runtime import QiskitRuntimeService, Session, EstimatorV2 as Estimator
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import Optimize1qGatesDecomposition

In [290]:
QiskitRuntimeService.save_account(
    channel='ibm_quantum',
    token='d8fca427febc152918d2b798bcb0f88fd2f74531f709378e610efd03ac7de61e44feedf887ebd2f159543cbf7025280bd214812b94d06c4435817a7bd07c5eea',
    set_as_default=True,
    overwrite=True,
)
service = QiskitRuntimeService()
real_backend = service.backend("ibm_brisbane")
aer_simulator = AerSimulator.from_backend(real_backend)

In [291]:
h_0 = 1
args = {'h_0': h_0}

def transverse_field_ising_hamiltonian(num_qubits):
    J = 1.0
    h_0 = 1.0
    graph = nx.path_graph(num_qubits)
    sparse_list = []
    
    for qubit in graph.nodes():
        coeff = ('X', [qubit], -1 * h_0)
        sparse_list.append(coeff)

    for i, j in graph.edges():
        coeff = ('ZZ', [i, j], -1 * J)
        sparse_list.append(coeff)
    
    hamiltonian = SparsePauliOp.from_sparse_list(sparse_list, num_qubits=num_qubits)
    return hamiltonian

In [292]:
def extract_operators(hamiltonian):
    operators = []
    for pauli_string, coeff in zip(hamiltonian.paulis, hamiltonian.coeffs):
        operators.append(SparsePauliOp(pauli_string))
    return operators

In [293]:
def separate_hamiltonians(operators):
    H_s_terms = []
    H_t_terms = []
    
    for op in operators:
        labels = op.paulis.to_labels()[0]  
        if 'Z' in labels and labels.count('Z') == 2:
            H_s_terms.append(op)
        elif 'X' in labels:
            H_t_terms.append(op)
    
    H_s = sum(H_s_terms) if H_s_terms else None
    H_t = H_t_terms if H_t_terms else []
    
    return H_s, H_t

In [294]:
def correct_matrix(E_matrix, inv_cond=1e-10):
    e_vals, e_vecs = np.linalg.eigh(E_matrix)
    e_vals_corrected = np.maximum(e_vals, inv_cond)
    E_matrix_corrected = e_vecs @ np.diag(e_vals_corrected) @ e_vecs.T.conj()
    E_inv = np.linalg.inv(E_matrix_corrected)
    return E_matrix_corrected, E_inv

In [295]:
def normalize_operator(op):
    op_data = op.to_matrix()
    norm = np.linalg.norm(op_data)
    if norm > 1e-10:
        return SparsePauliOp(op.paulis, op.coeffs / norm)
    return op

In [296]:
def is_valid_operator(op):
    op_data = op.to_matrix()
    return not np.any(np.isinf(op_data)) and not np.any(np.isnan(op_data)) and np.linalg.norm(op_data) > 1e-10

In [297]:
def commutator(op1, op2):
    return op1.compose(op2) - op2.compose(op1)

In [298]:
def generate_unique_commutators(operators):
    commutators = operators[:]
    seen_operators = set()

    def add_if_unique(op):
        op = normalize_operator(op)
        op_data = np.real_if_close(op.to_matrix())
        if is_valid_operator(op):
            op_tuple = tuple(op_data.flatten())
            if op_tuple not in seen_operators:
                seen_operators.add(op_tuple)
                return True
        return False

    while True:
        current_commutators = []
        for i, op1 in enumerate(commutators):
            for j, op2 in enumerate(commutators):
                if i < j:
                    comm = commutator(op1, op2)
                    comm = normalize_operator(comm)
                    if is_valid_operator(comm) and add_if_unique(comm):
                        current_commutators.append(comm)

        if not current_commutators:
            break

        commutators.extend(current_commutators)

    return commutators

In [299]:
def generate_operator_products(operators, K, max_products=None):
    operator_products = [SparsePauliOp(Pauli("I" * operators[0].num_qubits))] 
    for k in range(1, K + 1):
        for combo in combinations(operators, k):
            product_op = combo[0]
            for op in combo[1:]:
                product_op = product_op.compose(op)  
            operator_products.append(product_op)
    return operator_products

In [300]:
def is_independent(state, basis_states):
    flattened_states = np.array([s.data.flatten() for s in basis_states]).T
    state_flat = state.data.flatten()
    augmented_matrix = np.column_stack((flattened_states, state_flat))
    rank = np.linalg.matrix_rank(augmented_matrix)
    return rank > len(basis_states)

In [301]:
def apply_operators_to_state(initial_state, operator_products):
    basis_states = [initial_state]
    
    for op in operator_products[1:]:
        transformed_state = initial_state.evolve(op)
        if is_independent(transformed_state, basis_states):
            basis_states.append(transformed_state)
            
    return basis_states

In [302]:
def calculate_initial_alpha(initial_state, basis_states):
    initial_alpha = np.zeros(len(basis_states), dtype=np.complex128)
    
    if not isinstance(initial_state, Statevector):
        initial_state = Statevector(initial_state)
    
    for i, basis_state in enumerate(basis_states):
        if not isinstance(basis_state, Statevector):
            basis_state = Statevector(basis_state)
        
        initial_alpha[i] = np.vdot(initial_state.data, basis_state.data)    
    return initial_alpha

In [303]:
def transverse_field_coeff(t, args):
    return args['h_0'] * np.sin(2 * np.pi * t)

In [304]:
def evolve(t, y, E, D_s, D_t, args):
    
    f_t = transverse_field_coeff(t, args)
    
    D_timedependent = D_s + f_t * D_t

    if D_timedependent.shape[0] != D_timedependent.shape[1]:
        raise ValueError("D_timedependent não é uma matriz quadrada. Verifique D_s e D_t.")

    
    if y.shape[0] != D_timedependent.shape[0]:
        raise ValueError("Dimensão do vetor y não é compatível com D_timedependent.")

    
    dydt = -1j * np.linalg.solve(E, D_timedependent @ y)
    
    return dydt

In [305]:
def time_evolution_solver(H, initial_state, times, observable, args):
    result = qtp.sesolve(H, initial_state, times, e_ops=[observable], args=args)
    return result.expect[0]

In [306]:
estimator = Estimator(mode=aer_simulator)

In [307]:
def calculate_E_matrix(circuit, basis_states, estimator):
    n = len(basis_states)
    E = np.zeros((n, n), dtype=np.complex128)

    for i, op_i in enumerate(basis_states):
        for j, op_j in enumerate(basis_states):
            pub_E = (circuit, [op_i.compose(op_j)], [])
            job_E = estimator.run([pub_E])
            result_E = job_E.result()
            E[i, j] = result_E[0].data.evs[0]
    return E

def calculate_D_s_matrix(circuit, basis_states, H_s, estimator):
    n = len(basis_states)
    D_s = np.zeros((n, n), dtype=np.complex128)
    for i, op_i in enumerate(basis_states):
            for j, op_j in enumerate(basis_states):
                pub_Hs = (circuit, [op_i.compose(H_s).compose(op_j)], [])
                job_Hs = estimator.run([pub_Hs])
                result_Hs = job_Hs.result()
                D_s[i, j] = result_Hs[0].data.evs[0]
    return D_s

def calculate_D_t_matrix(circuit, basis_states, H_t, estimator):
    n = len(basis_states)
    D_t = np.zeros((n, n), dtype=np.complex128)
     
    for i, op_i in enumerate(basis_states):
        for j, op_j in enumerate(basis_states):
            result_Ht_sum = 0
            for H_t_op in H_t:
                    pub_Ht = (circuit, [op_i.compose(H_t_op).compose(op_j)], [])
                    job_Ht = estimator.run([pub_Ht])
                    job_Ht_result = job_Ht.result()
                    result_Ht_sum += job_Ht_result[0].data.evs[0]
            D_t[i, j] = result_Ht_sum
    return D_t

def calculate_D_matrix(D_s, D_t, times):
    n = D_s.shape[0]
    D = np.zeros((n, n, len(times)), dtype=np.complex128)
    
    for t_idx, t in enumerate(times):
        f_t = np.sin(2 * np.pi * t)
        D[:, :, t_idx] = D_s + f_t * D_t
    
    return D

def calculate_Z2_expectation_value(circuit, basis_states, alphas_t, Z2, times, estimator):
    n = len(basis_states)
    expectation_values = []

    for t_idx, t in enumerate(times):
        Z2_expectation = 0
        for i in range(n):
            for j in range(n):
                alpha_i = alphas_t[i, t_idx]
                alpha_j = alphas_t[j, t_idx].conjugate()
            
                pub_Z2 = (circuit, [basis_states[i].compose(Z2).compose(basis_states[j])], [])
                job_Z2 = estimator.run([pub_Z2])
                result_Z2 = job_Z2.result[0].data.evs[0]
            
                Z2_expectation += alpha_j * result_Z2 * alpha_i
    
        expectation_values.append(np.real(Z2_expectation))
    return expectation_values

In [308]:
num_qubits = 2
hamiltonian = transverse_field_ising_hamiltonian(num_qubits)
operators = extract_operators(hamiltonian)
unique_operators = generate_unique_commutators(operators)

In [309]:
operator_products = generate_operator_products(unique_operators, 1, max_products=100000)
H_s, H_t = separate_hamiltonians(operators)

In [310]:
circuit = QuantumCircuit(num_qubits)
initial_state = Statevector.from_instruction(circuit)
basis_states, independent_operators = apply_operators_to_state(initial_state, operators)

In [311]:
print("Operadores independentes gerados:")
for op in basis_states:
    print(op)

Operadores independentes gerados:
(1+0j)
0j
0j
0j


In [312]:
initial_alpha = calculate_initial_alpha(initial_state, basis_states)
initial_alpha /= np.linalg.norm(initial_alpha)

QiskitError: 'Invalid input data format for Statevector'

In [285]:
times = np.linspace(0, 8, 200)

In [286]:
E_raw = calculate_E_matrix(circuit, basis_states, estimator)
E, E_inv = correct_matrix(E_raw)
D_s = calculate_D_s_matrix(circuit, basis_states, H_s, estimator)
D_t = calculate_D_t_matrix(circuit, basis_states, H_t, estimator)
D = calculate_D_matrix(D_s, D_t, times)

In [287]:
print("Dimensão de initial_alpha:", initial_alpha.shape)
print("Dimensão de E:", E.shape)
print("Dimensão de D_s:", D_s.shape)
print("Dimensão de D_t:", D_t.shape)
print("Dimensão de args:", args)

Dimensão de initial_alpha: (3,)
Dimensão de E: (2, 2)
Dimensão de D_s: (2, 2)
Dimensão de D_t: (2, 2)
Dimensão de args: {'h_0': 1}


In [288]:
n = len(independent_operators)
if E.shape != (n, n) or D_s.shape != (n, n) or D_t.shape != (n, n):
    raise ValueError("As dimensões de E, D_s e D_t não são compatíveis com o número de operadores independentes.")
if initial_alpha.shape[0] != n:
    raise ValueError("Dimensão do vetor initial_alpha não é compatível com o número de operadores independentes.")


ValueError: Dimensão do vetor initial_alpha não é compatível com o número de operadores independentes.

In [135]:
sol = solve_ivp(evolve, (times[0], times[-1]), initial_alpha, t_eval=times, args=(E, D_s, D_t, args))

ValueError: Dimensão do vetor y não é compatível com D_timedependent.

In [ ]:
Z2 = SparsePauliOp.from_list([('IZ', 1)])

In [ ]:
H_exact = [H_s] + [[term, transverse_field_coeff] for term in H_t]
expectation_values_exact = time_evolution_solver(H_exact, initial_state, times, Z2, args)
expectation_values = calculate_Z2_expectation_value(circuit, basis_states, sol.y, Z2, times, estimator)

In [ ]:
plt.plot(times, expectation_values_exact, label="Exact", color='red')
plt.plot(times, expectation_values, 'o', label="QAS", color='blue', markevery=5)
plt.xlabel("Time")
plt.ylabel(r'$\langle Z_{2} \rangle$')
plt.legend()
plt.show()